In [48]:
import pandas as pd
import sqlite3

1. Create a connection to the database using the sqlite3 library.

In [65]:
db_file = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_file)

2. Create a new table called datamart in the database by joining the tables pageviews and checker using only one query.
- The table should have the following columns: "uid", "labname", "first_commit_ts", and "first_view_ts".
- "first_commit_ts" is a new name for the "timestamp" column in the checker table. It shows the first commit from a particular lab and user.
- "first_view_ts" shows the first time a user visited the pageviews table. It is the timestamp of when a user visited the newsfeed.
- status = 'ready' should still be a filter.
- numTrials = 1 should still be a filter.
- "labnames" should be from the list: laba04, laba04s, laba05, laba06, laba06s, and project1.
- The table should contain only users (uids with user_*), not admins.
- "first_commit_ts" and "first_view_ts" should be parsed as datetime64[ns].

In [50]:
create_datamart_query = """
CREATE TABLE datamart AS
SELECT 
    c.uid,
    c.labname,
    c.first_commit_ts,
    p.first_view_ts
FROM (
    SELECT 
        uid,
        labname,
        MIN(timestamp) AS first_commit_ts
    FROM checker
    WHERE status = 'ready'
      AND numTrials = 1
      AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
      AND uid LIKE 'user_%'
    GROUP BY uid, labname
) c
LEFT JOIN (
    SELECT 
        uid,
        MIN(datetime) AS first_view_ts
    FROM pageviews
    WHERE uid LIKE 'user_%'
    GROUP BY uid
) p ON c.uid = p.uid;
"""
cursor = conn.cursor()
cursor.execute(create_datamart_query)
conn.commit()



3. Using Pandas methods, create two dataframes: test and control.
- test should contain users with values in "first_view_ts".
- control should contain users with missing values in "first_view_ts".
- Replace the missing values in the control table with the average "first_view_ts" value of the test users. We will use this value for future analyses.
- Save both tables in the database. You will use them in the next exercises.

In [57]:
get_data_mart = "SELECT * from datamart"
datamart = pd.read_sql(get_data_mart, conn, parse_dates=['first_commit_ts', 'first_view_ts'])

In [58]:
datamart.head(15)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [52]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

In [59]:
test.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136


In [60]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,NaT
13,user_11,project1,2020-05-03 23:45:33.673409,NaT
14,user_12,laba04,2020-04-18 17:07:51.767358,NaT
15,user_12,laba04s,2020-04-26 15:42:38.070593,NaT
16,user_12,laba05,2020-05-03 08:39:25.174316,NaT


In [62]:
user_avg = test.drop_duplicates('uid')['first_view_ts'].mean()
control['first_view_ts'] = user_avg
control.head()

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-26 11:42:34.626102
13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-26 11:42:34.626102
14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-26 11:42:34.626102
15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-26 11:42:34.626102
16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-26 11:42:34.626102


In [74]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)

81

4. Close the connection.

In [76]:
conn.close()